### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provids the richest features set with field validation, description, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021CD233F2F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021CD233E840>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import Field, BaseModel

class Movie(BaseModel):
    #this is the structured output that LLM have to make
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This is year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies ratings out of 10")

In [4]:
model_with_struture = model.with_structured_output(Movie)
model_with_struture

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021CD233F2F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021CD233E840>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [5]:
model.invoke("Provide details about the movie intersteller")

AIMessage(content='**Interstellar** (2014) – A Sci‑Fi Epic of Space, Time, and Human Hope  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher Nolan |\n| **Screenwriters** | Christopher Nolan, Jonathan Nolan |\n| **Producers** | Christopher Nolan, Emma Thomas, Lynda Obst |\n| **Production Companies** | Syncopy, Legendary Pictures, Warner Bros. Pictures |\n| **Budget** | $165\u202fmillion (est.) |\n| **Box Office** | $677\u202fmillion worldwide |\n| **Release Dates** | • UK: 5\u202fNov\u202f2014 (Cineworld)  <br>• US: 7\u202fNov\u202f2014 (Theatrical) |\n| **Runtime** | 169\u202fminutes (2\u202fh\u202f49\u202fmin) |\n| **Language** | English |\n| **Genres** | Science‑fiction, Adventure, Drama |\n| **Cinematography** | Hoyte van Hoytema |\n| **Music** | Hans Zimmer |\n| **Special Effects** | ILM, Digital Domain, Weta Digital, Double Negative |\n| **Notable Awards** | 3‑time Academy Award nominations (Best Visual Effects, Best Sound Editing, Best Sound Mixing)

In [6]:
model_with_struture.invoke("Provide details about the movie intersteller")

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

### Message output alongside parsed structure

In [ ]:
from pydantic import Field, BaseModel

class Movie(BaseModel):
    """A movie with details."""
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="This is year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movies ratings out of 10")
    
model_with_struture = model.with_structured_output(Movie, include_raw=True)

response = model_with_struture.invoke("provide the details about the movie jurassic park")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "provide the details about the movie jurassic park". We have a tool function "functions.Movie" that expects director, rating, title, year. We need to call it with those arguments. Provide details about Jurassic Park: director: Steven Spielberg, rating: maybe 8.1 (IMDB rating), title: "Jurassic Park", year: 1993. Use the tool.', 'tool_calls': [{'id': 'fc_8946927f-0829-42cc-bfac-7aa65adbc8a1', 'function': {'arguments': '{"director":"Steven Spielberg","rating":8.1,"title":"Jurassic Park","year":1993}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 166, 'total_tokens': 290, 'completion_time': 0.183729603, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.009532925, 'prompt_tokens_details': None, 'queue_time': 0.284443783, 'total_time': 0.193262528}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e

### Nested structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million dollars USD")
    
model_with_struture = model.with_structured_output(MovieDetails)
response = model_with_struture.invoke("provide detail about movie the avengers")
response

MovieDetails(title='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actor(name='Tom Hiddleston', role='Loki')], genres=['Action', 'Adventure', 'Science Fiction'], budget=220000000.0)

### TypeDict

TypeDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import TypedDict,  Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]
    
model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("provide the details of the movie hungama")
response

{'director': 'Priyadarshan', 'rating': 6.5, 'title': 'Hungama', 'year': 2003}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using @dataclass decorator

In [10]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")
    
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo,  # Uses the Google GenAI provider
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-45678",
            }
        ]
    }
)
result["structured_response"]



ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-45678')

In [18]:
## Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str
    email: str
    phone: str 
    
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo,  # Uses the Google GenAI provider
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-45678",
            }
        ]
    }
)
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-45678')